In [1]:
import numpy as np
import pandas as pd
import sys
sys.path.append(".")

from state_coords import STATE_COORDS
from geo_utils import haversine

np.random.seed(42)

RAW_PATH = "../data/Nassau_Candy_Distributor.csv"
OUT_PATH = "../outputs/cleaned_data.csv"

LEAD_TIME_RANGES = {
    "Same Day": (0, 1),
    "First Class": (1, 3),
    "Second Class": (3, 5),
    "Standard Class": (5, 8),
}

In [2]:
df = pd.read_csv(RAW_PATH)
factories = pd.read_csv("../data/factories.csv")
pf_map = pd.read_csv("../data/product_factory_map.csv")
print(df.shape)
df.head()

(10194, 18)


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Country/Region,City,State/Province,Postal Code,Division,Region,Product ID,Product Name,Sales,Units,Gross Profit,Cost
0,1,US-2021-103800-CHO-MIL-31000,03-01-2024,30-06-2026,Standard Class,103800,United States,Houston,Texas,77095,Chocolate,Interior,CHO-MIL-31000,Wonka Bar - Milk Chocolate,6.50,2,4.22,2.28
1,2,US-2021-112326-CHO-TRI-54000,04-01-2024,01-07-2026,Standard Class,112326,United States,Naperville,Illinois,60540,Chocolate,Interior,CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,7.50,2,4.90,2.60
2,3,US-2021-112326-CHO-NUT-13000,04-01-2024,01-07-2026,Standard Class,112326,United States,Naperville,Illinois,60540,Chocolate,Interior,CHO-NUT-13000,Wonka Bar - Nutty Crunch Surprise,10.47,3,7.47,3.00
3,4,US-2021-112326-CHO-SCR-58000,04-01-2024,01-07-2026,Standard Class,112326,United States,Naperville,Illinois,60540,Chocolate,Interior,CHO-SCR-58000,Wonka Bar -Scrumdiddlyumptious,10.80,3,7.50,3.30
4,5,US-2021-141817-CHO-TRI-54000,05-01-2024,05-07-2026,Standard Class,141817,United States,Philadelphia,Pennsylvania,19143,Chocolate,Atlantic,CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,11.25,3,7.35,3.90


In [3]:
df = df.merge(pf_map[["Product Name", "Factory"]], on="Product Name", how="left")
df = df.merge(factories, on="Factory", how="left")
df = df.rename(columns={"Latitude": "Factory_Lat", "Longitude": "Factory_Lon"})

print("Missing factory mapping:", df["Factory"].isna().sum())

Missing factory mapping: 0


In [4]:
coords = df["State/Province"].map(STATE_COORDS)
print("Missing state coordinate:", coords.isna().sum())

df["Dest_Lat"] = coords.apply(lambda x: x[0])
df["Dest_Lon"] = coords.apply(lambda x: x[1])

df["Shipping_Distance_KM"] = haversine(
    df["Factory_Lat"], df["Factory_Lon"], df["Dest_Lat"], df["Dest_Lon"]
)
df[["Product Name", "Factory", "Shipping_Distance_KM"]].sample(5)

Missing state coordinate: 0


,Product Name,Factory,Shipping_Distance_KM
6636,Wonka Bar - Fudge Mallows,Lot's O' Nuts,2704.965207
3742,Wonka Bar - Triple Dazzle Caramel,Wicked Choccy's,412.511932
5114,Wonka Bar - Triple Dazzle Caramel,Wicked Choccy's,1160.081298
6505,Wonka Bar - Fudge Mallows,Lot's O' Nuts,809.097829
10072,Wonka Bar -Scrumdiddlyumptious,Lot's O' Nuts,1861.250505


In [6]:
def simulate_lead_time(ship_mode_series, distance_series):
    lead_times = np.zeros(len(ship_mode_series))

    for mode, (lo, hi) in LEAD_TIME_RANGES.items():
        mask = (ship_mode_series == mode).values
        n = mask.sum()
        if n == 0:
            continue
        mode_point = lo + (hi - lo) * 0.3
        sampled = np.random.triangular(lo, mode_point, hi, n)
        lead_times[mask] = sampled

    # distance penalty: up to +1 extra day at the farthest realistic distance
    max_dist = distance_series.max()
    distance_penalty = (distance_series / max_dist) * 1.0

    noise = np.random.normal(0, 0.3, len(lead_times))
    lead_times = np.clip(lead_times + distance_penalty.values + noise, 0, 12)
    return np.round(lead_times, 2)

In [7]:
df["Lead Time"] = simulate_lead_time(df["Ship Mode"], df["Shipping_Distance_KM"])
df.groupby("Ship Mode")["Lead Time"].describe()

,count,mean,std,min,25%,50%,75%,max
Ship Mode,,,,,,,,
First Class,1548.0,2.277422,0.562936,0.29,1.87,2.25,2.6625,4.05
Same Day,547.0,0.878739,0.428705,0.00,0.58,0.86,1.1800,2.35
Second Class,1979.0,4.277655,0.564160,2.38,3.88,4.26,4.6600,6.00
Standard Class,6120.0,6.715757,0.724143,4.65,6.20,6.66,7.2100,9.07


In [8]:
for col in ["Sales", "Cost", "Gross Profit"]:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - 3 * iqr, q3 + 3 * iqr
    before = len(df)
    df = df[(df[col] >= lo) & (df[col] <= hi)]
    removed = before - len(df)
    print(f"Removed {removed} outlier rows on {col}")

Removed 72 outlier rows on Sales
Removed 66 outlier rows on Cost
Removed 11 outlier rows on Gross Profit


In [9]:
df["Profit_Margin"] = df["Gross Profit"] / df["Sales"]
df["Cost_Per_Unit"] = df["Cost"] / df["Units"]

df[["Sales", "Gross Profit", "Profit_Margin", "Cost", "Units", "Cost_Per_Unit"]].describe()

,Sales,Gross Profit,Profit_Margin,Cost,Units,Cost_Per_Unit
count,10045.000000,10045.000000,10045.000000,10045.000000,10045.000000,10045.000000
mean,13.103870,8.783803,0.667765,4.320067,3.742260,1.159944
std,7.590325,5.184803,0.060332,2.538035,2.140051,0.265880
min,1.250000,0.250000,0.076923,0.600000,1.000000,0.600000
25%,7.200000,4.900000,0.653333,2.400000,2.000000,1.100000
50%,10.800000,7.470000,0.666667,3.600000,3.000000,1.140000
75%,18.000000,12.250000,0.694444,5.700000,5.000000,1.200000
max,46.800000,32.500000,0.800000,15.600000,13.000000,10.000000


In [10]:
df.to_csv(OUT_PATH, index=False)
print(f"Saved cleaned dataset: {df.shape}")

Saved cleaned dataset: (10045, 27)
